# H2: PP-OCRv6 small: оправдана ли дополнительная ёмкость?

**Проверяемая гипотеза:** Прирост exact match к tiny минимум 2 п.п., p95 не выше 150 мс.

LCNetV4 → LightSVTR с локальными свёртками и self-attention → CTC. В отличие от tiny есть отдельное моделирование контекста последовательности. Меняются одновременно ёмкость, словарь, детали обучения и веса; это сравнение готовых моделей.

[Теория и исследования](../research/THEORY_AND_PAPERS.md). [Зафиксированный протокол](../research/PROTOCOL_V2.md). Пороговые значения — наши инженерные критерии, не требования заказчика.

In [1]:
from pathlib import Path
import sys, json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'captcha_lab').is_dir())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
import pandas as pd
from captcha_lab.evaluation_v2 import (manifest,run_local,run_gemini,metrics,scored,
                                      compare,all_summaries,hypothesis_results,RESULTS)
audit=json.loads((ROOT/'datasets/captcha-images/audit.json').read_text())
assert audit['holdout_images']==1984 and audit['shared_images']==200
display(pd.DataFrame([{'split':k,'n':v} for k,v in audit['source_split_counts'].items()]))
print('Holdout manifest SHA-256:',audit['holdout_manifest_sha256'])
print('Shared manifest SHA-256:',audit['shared_manifest_sha256'])
print('No prediction caches: inference cell below executes the actual model.')


,split,n
0,train,6000
1,validation,2000
2,test,2000


Holdout manifest SHA-256: 33fe043e8dec7b0e238c6bc81bb63c601699540b4b6c2025ad9480e21164f42b
Shared manifest SHA-256: 675bc9b4d93369e68817412fa08a72c6e21de23c820e00e42bc30d06c7390384
No prediction caches: inference cell below executes the actual model.


## Размеченная выборка

Публичный полный test содержит 2000 изображений; 16 просмотренных в предыдущем пилоте исключены. Метки и архивы скачаны полностью, не из preview. Функция распознавания получает только пиксели; метки присоединяются после предсказания. Исходные семь целевых меток выделены отдельно, два спорных примера исключены.

In [2]:
labels=manifest('shared_200' if 'ppocr-small'=='gemini' else 'holdout')
display(labels[['image_id','label','image_path']].head(12))
print('Evaluation images:',len(labels))

,image_id,label,image_path
0,test_000c1b6d4e5f11fb75d55a7c094b901e,161660,data/benchmark/images/test/161660.000c1b6d4e5f...
1,test_00105a4c053053509955d95368312987,496670,data/benchmark/images/test/496670.00105a4c0530...
2,test_001a3b40ef793b5984df8969f145d7bb,744782,data/benchmark/images/test/744782.001a3b40ef79...
3,test_0021e642491cf5bbc13cfc59fa46084a,017232,data/benchmark/images/test/017232.0021e642491c...
4,test_0047646a0e3b67ebc360e43ba88d1596,853091,data/benchmark/images/test/853091.0047646a0e3b...
5,test_0064b2188592a16991d5394b3ec5bf01,959686,data/benchmark/images/test/959686.0064b2188592...
6,test_00a5d81ab5060a0d45608169ce20bda9,159645,data/benchmark/images/test/159645.00a5d81ab506...
7,test_00e02c7c9da2cc0d3cdf5e3d9183a9f5,952376,data/benchmark/images/test/952376.00e02c7c9da2...
8,test_0123f4327a176c460aeefecd1d69bc11,374639,data/benchmark/images/test/374639.0123f4327a17...
9,test_0127972a796c1f306fdb5f919a7649fa,517493,data/benchmark/images/test/517493.0127972a796c...


Evaluation images: 1984


## Реальный прогон

Ячейка заново загружает модель и выполняет inference на каждой картинке. Ошибки сохраняются, готовые ответы не подставляются. CPU, batch=1, отдельный прогрев на validation.

In [3]:
frames=run_local('ppocr-small')

ppocr-small holdout 200 / 1984


ppocr-small holdout 400 / 1984


ppocr-small holdout 600 / 1984


ppocr-small holdout 800 / 1984


ppocr-small holdout 1000 / 1984


ppocr-small holdout 1200 / 1984


ppocr-small holdout 1400 / 1984


ppocr-small holdout 1600 / 1984


ppocr-small holdout 1800 / 1984


In [4]:
rows=[]
for dataset,predictions in frames.items():
    labels=manifest(dataset)
    rows.append(dict(dataset=dataset,**metrics(predictions,labels)))
    joined=scored(predictions,labels)
    destination=RESULTS/('ppocr-small')/(dataset+'_scored.csv')
    joined.to_csv(destination,index=False)
    print(dataset,'— ошибки по изображениям:')
    display(joined.loc[~joined.correct,['image_id','label','prediction','error','error_detail']].head(20))
display(pd.DataFrame(rows))


holdout — ошибки по изображениям:


,image_id,label,prediction,error,error_detail
40,test_059a19ca75774a6ff857551a862a9eb9,772537,712537,,
55,test_0793dc242a40b91872d09106dea3cb64,173367,73367,,
72,test_0a5a2dad9d0c93d13c721af538dc74f3,677799,77799,,
84,test_0babecfa8fc8a18b3726c0ef3d5197d1,844073,844073-,,
85,test_0bb791f343df3cd5fe35ec5f6be942df,071887,71887,,
131,test_10aa0ed8771dcd750c1b1e26f00e6af2,538261,538261.,,
143,test_112da6d0e0769450fbc3248d7681f1ea,594026,594026.,,
150,test_11a26c470c305ef4f9922e36ef7c1635,579877,579877.,,
151,test_11e687de5eca9c8e1fee48660bd2c497,020684,20684,,
153,test_121e092360cac568f640e4e048d1470b,592494,392494,,


target — ошибки по изображениям:


,image_id,label,prediction,error,error_detail
0,33ef91d464c03eb9,894412,8048,,
1,41ada5001f1e1e88,432077,432071,,
5,d4c0652824fb2dfd,639125,60926,,


,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
0,holdout,1984,1876,0.945565,0.934693,0.954714,0.010165,0,0,25.2850,30.772065,25.2850,30.772065
1,target,7,4,0.571429,0.250458,0.841780,0.190476,0,0,29.4816,32.536460,29.4816,32.536460


## Вывод по гипотезе

Exact match и CER вычисляются по скачанным меткам. Вердикт получается из заранее заданного критерия и интервала разности; отсутствие статистической определённости явно показывается. Правильность на публичном наборе не переносится автоматически на Instagram.

In [5]:
summary=all_summaries()
display(summary[summary.model.eq('ppocr-small')])
hypotheses,paired=hypothesis_results()
display(hypotheses[hypotheses.hypothesis.eq('H2')])
display(paired[paired.hypothesis.eq('H2')] if not paired.empty else paired)

,model,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
3,ppocr-small,holdout,1984,1876,0.945565,0.934693,0.954714,0.010165,0,0,25.2850,30.772065,25.2850,30.772065
4,ppocr-small,shared_200,200,188,0.940000,0.898068,0.965348,0.010833,0,0,25.3915,30.484690,25.3915,30.484690
5,ppocr-small,target,7,4,0.571429,0.250458,0.841780,0.190476,0,0,29.4816,32.536460,29.4816,32.536460


,hypothesis,criterion,verdict,estimate,ci_low,ci_high
1,H2,delta ppocr-small-ppocr-tiny>=2.00%,supported,0.454637,0.425023,0.481954


,hypothesis,model_a,model_b,n,wins,losses,delta,delta_ci95_low,delta_ci95_high,mcnemar_exact_p,holm_p
0,H2,ppocr-small,ppocr-tiny,1984,910,8,0.454637,0.425023,0.481954,1.104736e-257,1.104736e-257
